# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a complete walk-through for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
# Import libraries
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Dataset version: {getattr(dataset.metadata, 'version', 'N/A')}")
print(f"Published on: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(dataset.metadata, 'license', 'N/A')}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below we enumerate all record sets, their fields, and field (column) `@id`s. All references use Croissant `@id` fields for robust referencing.

In [ ]:
# Helper function to recursively print records, fields, and columns with their @id
from mlcroissant._src.structure.graph.dataset import Field, RecordSet

def print_structure(ds):
    print('Record Sets:')
    record_sets = ds.metadata.record_sets
    for i, rset in enumerate(record_sets):
        print(f"  [{i}] Record set name: '{rset.name}', @id: '{rset.id_}'")
        print('    Fields:')
        for j, fld in enumerate(rset.fields):
            print(f"      - [{j}] Field name: '{fld.name}', @id: '{fld.id_}' (dataType: {fld.data_type})")
            if fld.columns:
                for k, col in enumerate(fld.columns):
                    print(f"           Column: '{col.name}', @id: '{col.id_}' (encodingFormat: {getattr(col, 'encoding_format', None)})")
        print()
print_structure(dataset)

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis, referencing entities by their `@id` as required by Croissant. Choose which record set(s) you would like to explore based on the overview above.

In [ ]:
# List available record set @id's
record_sets = [rset.id_ for rset in dataset.metadata.record_sets]
print('RecordSet @id\'s:', record_sets)

# Example: Use the first available record set for demonstration (change as needed!)
selected_record_set_id = record_sets[0] if record_sets else None
print('Using record set:', selected_record_set_id)

dataframes = {}
for record_set in record_sets:
    print(f"Loading records for RecordSet @id: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if len(records) == 0:
        print(f"  No records found in '{record_set}'.")
    else:
        df = pd.DataFrame(records)
        print(f"  Columns: {list(df.columns)}\n  Number of records: {len(df)}")
        dataframes[record_set] = df

# If you want to continue with the first available record set containing records:
for rec_id, df in dataframes.items():
    if len(df) > 0:
        default_rs_id = rec_id
        break
else:
    default_rs_id = None
if default_rs_id:
    print(f"\nDefault record set selected for EDA: {default_rs_id}")
    print('Field columns:', list(dataframes[default_rs_id].columns))
    display(dataframes[default_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll now conduct typical EDA on the dataset, working with fields (columns) referenced by Croissant `@id`. 

The following example selects a numeric field, filters rows above a threshold, normalizes the field and groups by a user-selected field.

In [ ]:
# Choose a numeric field for analysis based on the preceding cell output
# You should select the actual @id for a numeric field as found above.
# For example, let's try to use 'coverage_percent' or similar if available. Update if needed.

if default_rs_id:
    df = dataframes[default_rs_id]
    # Try to select a likely numeric field
    numeric_fields = [col for col in df.columns if df[col].dtype in [int, float, 'int64', 'float64', 'float32', 'int32'] or pd.api.types.is_numeric_dtype(df[col])]
    if len(numeric_fields) == 0:
        # Try to coerce all possible numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                continue
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    print('Numeric fields available:', numeric_fields)
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std())
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a categorical field if one exists
        categorical_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        print('Categorical fields available:', categorical_fields)
        if categorical_fields:
            group_field = categorical_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean({numeric_field}) by '{group_field}':")
            display(grouped_df.head())
    else:
        print('No numeric fields available for EDA.')
else:
    print('No records available in any record set for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example histogram and boxplot for the selected numeric field, and a bar chart grouped by the first available categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization provided only if records and a numeric field are available
if default_rs_id and 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Histogram of {numeric_field}")

    plt.subplot(1,2,2)
    sns.boxplot(x=filtered_df[numeric_field])
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()

    # If we grouped by a categorical field, show barplot
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No records available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a FAIR² dataset described by a Croissant schema using the `mlcroissant` library. We referenced record sets, fields, and columns by their Croissant `@id`, explored the structure and metadata, extracted records, performed EDA on numeric fields, and visualized data distributions. 

- All entity references (record sets, fields, columns) use their `@id` for robustness and reproducibility.
- Steps here can be adapted to new record sets and fields by updating the `@id` values as needed, consistent with the FAIR and Croissant best practices.

Continue exploring by adapting the filtering, grouping, and visualization logic to the specific biological or experimental metadata within this (or other) Croissant datasets!